In [ ]:
import os, sys, glob

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
WORK = "/content" if IN_COLAB else "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

!pip install -q faster-whisper spacy
!python -m spacy download en_core_web_sm -q

# مسار ASL Citizen: موجود في Kaggle فقط
ASL_ROOT = None
if IN_KAGGLE:
    hits = glob.glob("/kaggle/input/**/ASL_Citizen", recursive=True)
    ASL_ROOT = hits[0] if hits else None

print("البيئة:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "محلي")
print("مجلد العمل:", WORK)
print("ASL Citizen:", ASL_ROOT)     # في Colab: None وهذا المطلوب

النموذج والدوال لمرحلة الصوت

In [ ]:
import subprocess, re
import numpy as np
from faster_whisper import WhisperModel
import spacy

asr_model = WhisperModel("small.en", device="cpu", compute_type="int8")
nlp = spacy.load("en_core_web_sm")


def load_audio(path, sr=16000):
    """قراءة أي ملف صوتي وتحويله إلى مصفوفة numpy عبر ffmpeg"""
    cmd = ["ffmpeg", "-nostdin", "-loglevel", "error", "-i", path,
           "-f", "s16le", "-ac", "1", "-ar", str(sr), "-"]
    out = subprocess.run(cmd, capture_output=True, check=True).stdout
    return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0


def transcribe_audio(path):
    segments, _ = asr_model.transcribe(load_audio(path), language="en",
                                       beam_size=5, vad_filter=True)
    return " ".join(s.text.strip() for s in segments)


def clean_text(text):
    text = re.sub(r"\[.*?\]|\(.*?\)", "", text)
    text = text.replace("’", "'")
    return re.sub(r"\s+", " ", text).strip()


def split_sentences(text):
    return [s.text.strip() for s in nlp(text).sents if s.text.strip()]


print("جاهز ✅")

In [ ]:
from IPython.display import display, HTML
from google.colab import output
from base64 import b64decode

audio_file = f"{WORK}/recorded.webm"

def save_audio(data_url):
    with open(audio_file, "wb") as f:
        f.write(b64decode(data_url.split(",")[1]))

output.register_callback("notebook.save_audio", save_audio)

display(HTML("""
<button id="startBtn" style="font-size:16px;padding:8px 16px">🎙️ ابدأ التسجيل</button>
<button id="stopBtn" style="font-size:16px;padding:8px 16px" disabled>⏹️ إيقاف</button>
<p id="status">اضغط "ابدأ التسجيل" ثم تكلم</p>
<script>
let recorder, chunks = [];
const startBtn = document.getElementById("startBtn");
const stopBtn = document.getElementById("stopBtn");
const status = document.getElementById("status");
startBtn.onclick = async () => {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    chunks = [];
    recorder = new MediaRecorder(stream);
    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.onstop = () => {
      stream.getTracks().forEach(t => t.stop());
      const blob = new Blob(chunks, {type: recorder.mimeType});
      const reader = new FileReader();
      reader.onloadend = async () => {
        await google.colab.kernel.invokeFunction("notebook.save_audio", [reader.result], {});
        status.textContent = "✅ تم الحفظ، شغّل الخلية التالية";
      };
      reader.readAsDataURL(blob);
    };
    recorder.start();
    startBtn.disabled = true; stopBtn.disabled = false;
    status.textContent = "🔴 جارٍ التسجيل... تكلم الآن";
  } catch (err) { status.textContent = "❌ خطأ في الميكروفون: " + err; }
};
stopBtn.onclick = () => {
  recorder.stop();
  startBtn.disabled = false; stopBtn.disabled = true;
  status.textContent = "⏳ جارٍ الحفظ...";
};
</script>
"""))


In [ ]:
assert os.path.exists(audio_file), "لا يوجد ملف صوتي، شغّل الخلية 2ب أولاً"

raw_text = transcribe_audio(audio_file)
clean    = clean_text(raw_text)

print("الخام  :", raw_text)
print("المنظف :", clean)
for i, s in enumerate(split_sentences(clean), 1):
    print(f"{i}. {s}")

محرك الغلوس

In [ ]:
PRONOUNS = {
    "i": "ME", "me": "ME", "my": "MY", "mine": "MY", "myself": "MYSELF",
    "you": "YOU", "your": "YOUR", "yours": "YOUR",
    "he": "HE", "him": "HE", "his": "HIS",
    "she": "SHE", "her": "SHE",
    "we": "WE", "us": "WE", "our": "OUR",
    "they": "THEY", "them": "THEY", "their": "THEIR",
    "it": "IT", "its": "ITS",
}

TIME_WORDS = {
    "TOMORROW", "YESTERDAY", "TODAY", "NOW", "TONIGHT", "LATER",
    "MORNING", "AFTERNOON", "EVENING", "NIGHT",
    "MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY",
    "FRIDAY", "SATURDAY", "SUNDAY",
}
TIME_UNITS = {"WEEK", "MONTH", "YEAR", "DAY", "HOUR", "MINUTE"}
TIME_MODIFIERS = {"last", "next", "this", "every"}
PAST_MARKER, FUTURE_MARKER = "BEFORE", "FUTURE"
TENSE_MARKERS = {PAST_MARKER, FUTURE_MARKER}

MODAL_MAP = {"can": "CAN", "could": "CAN", "must": "MUST",
             "should": "SHOULD", "may": "MAYBE", "might": "MAYBE"}
MODAL_NEG = {"can": "CAN'T", "could": "CAN'T", "must": "MUST-NOT",
             "should": "SHOULD-NOT"}
FUTURE_MODALS = {"will", "shall"}

NEG_FUSION = {"like": "DON'T-LIKE", "want": "DON'T-WANT", "know": "DON'T-KNOW",
              "understand": "DON'T-UNDERSTAND", "need": "DON'T-NEED"}

WH_WORDS = {"what", "where", "when", "who", "whom", "why", "how", "which"}
WH_TAGS = {"WP", "WRB", "WDT", "WP$"}

DROP_POS = {"DET", "AUX", "PUNCT", "PART"}
DROP_WORDS = {"to", "of", "in", "at", "on", "is", "am", "are", "be",
              "the", "a", "an"}
KEEP_DET = {"this", "that", "these", "those", "no", "every",
            "all", "some", "each"}

NAME_ENTS = {"PERSON", "GPE", "ORG", "LOC", "FAC"}

LEXICON = {
    "mom": "MOTHER", "mommy": "MOTHER", "mum": "MOTHER",
    "dad": "FATHER", "daddy": "FATHER",
    "hi": "HELLO", "hey": "HELLO", "thanks": "THANK-YOU",
    "okay": "OK", "bye": "GOODBYE", "children": "CHILD",
    "people": "PEOPLE", "kids": "CHILD",
}

PHRASES = {
    ("thank", "you"): "THANK-YOU",
    ("good", "morning"): "GOOD-MORNING",
    ("good", "afternoon"): "GOOD-AFTERNOON",
    ("good", "night"): "GOOD-NIGHT",
    ("excuse", "me"): "EXCUSE-ME",
    ("how", "are", "you"): "HOW-ARE-YOU",
    ("nice", "to", "meet", "you"): "NICE-MEET-YOU",
}

CLAUSE_CONFIG = {
    "if": ("IF", "before"), "unless": ("UNLESS", "before"),
    "when": ("WHEN", "before"), "although": ("ALTHOUGH", "before"),
    "though": ("ALTHOUGH", "before"),
    "because": ("BECAUSE", "after"), "since": ("BECAUSE", "after"),
}
COORD = {"and": None, "but": "BUT", "or": "OR", "so": "SO"}

print("تم تحميل القواميس ✅")

In [ ]:
CONTRACTIONS = {
    r"\bgonna\b": "going to",
    r"\bwanna\b": "want to",
    r"\bgotta\b": "have to",
    r"\blemme\b": "let me",
    r"\bgimme\b": "give me",
    r"\bdunno\b": "do not know",
    r"\by'all\b": "you all",
    r"\bain't\b": "is not",
    r"(?:\bcuz\b|'cause\b)": "because",
}

FILLERS = r"\b(?:u+m+|u+h+|erm+|hmm+|mmm+|ah+)\b,?\s*"
STUTTER = r"\b(i|we|you|the|a|to|it|and|my)\s+\1\b"

UNITS = {"zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,
         "seven":7,"eight":8,"nine":9,"ten":10,"eleven":11,"twelve":12,
         "thirteen":13,"fourteen":14,"fifteen":15,"sixteen":16,
         "seventeen":17,"eighteen":18,"nineteen":19}
TENS = {"twenty":20,"thirty":30,"forty":40,"fifty":50,
        "sixty":60,"seventy":70,"eighty":80,"ninety":90}
SCALES = {"thousand":1_000, "million":1_000_000, "billion":1_000_000_000}

NUM_WORD = "|".join(list(UNITS) + list(TENS) + ["hundred"] + list(SCALES))
NUM_RUN = re.compile(
    rf"\b(?:{NUM_WORD})(?:(?:[\s-]+and)?[\s-]+(?:{NUM_WORD}))*\b", re.I)

ONE_PRONOUN_BEFORE = {"the", "this", "that", "which", "no", "any",
                      "each", "every", "another"}

def words_to_int(phrase):
    total, current = 0, 0
    for w in re.split(r"[\s-]+", phrase.lower()):
        if w == "and":
            continue
        if w in UNITS:
            current += UNITS[w]
        elif w in TENS:
            current += TENS[w]
        elif w == "hundred":
            current = (current or 1) * 100
        elif w in SCALES:
            total += (current or 1) * SCALES[w]
            current = 0
    return total + current

def _replace_numbers(text):
    def repl(m):
        phrase = m.group(0)
        if phrase.lower() == "one":
            before = text[:m.start()].split()
            after = text[m.end():].split()
            prev = before[-1].lower().strip(",.") if before else ""
            nxt = after[0].lower().strip(",.") if after else ""
            if prev in ONE_PRONOUN_BEFORE or nxt == "of":
                return phrase
        return str(words_to_int(phrase))
    return NUM_RUN.sub(repl, text)

def normalize_text(text):
    text = text.replace("’", "'")
    for pat, rep in CONTRACTIONS.items():
        text = re.sub(pat, rep, text, flags=re.I)
    text = re.sub(FILLERS, "", text, flags=re.I)
    text = re.sub(STUTTER, r"\1", text, flags=re.I)
    text = re.sub(r"(?<=\d),(?=\d{3}\b)", "", text)
    text = _replace_numbers(text)
    return re.sub(r"\s+", " ", text).strip()

print("تم تعريف التطبيع ✅")

In [ ]:
def base_gloss(t):
    w, lem = t.lower_, t.lemma_.lower()
    if w in PRONOUNS:
        return PRONOUNS[w]
    if w in WH_WORDS and t.tag_ in WH_TAGS:      # ← سطرا الإصلاح: كلمات الاستفهام لا تُحذف
        return w.upper()                         #   حتى لو صنّفها spaCy كأداة تعريف
    if lem.upper() in TIME_WORDS:
        return lem.upper()
    if t.pos_ == "PROPN" or t.ent_type_ in NAME_ENTS:
        return "fs-" + t.text.upper()
    if t.pos_ == "DET" and w in KEEP_DET:
        return w.upper()
    if t.pos_ in DROP_POS or w in DROP_WORDS:
        return None
    if t.like_num:
        return t.text.upper()
    return LEXICON.get(w) or LEXICON.get(lem) or lem.upper()


def analyze(sentence):
    sentence = normalize_text(sentence)          # التطبيع داخل المحوّل مباشرة
    doc = nlp(sentence)
    n = len(doc)
    skip, forced = set(), {}

    # (1) العبارات الثابتة و have to و going to + فعل
    words = [t.lower_ for t in doc]
    for i in range(n):
        for phrase, g in PHRASES.items():
            L = len(phrase)
            if tuple(words[i:i+L]) == phrase:
                forced[i] = g
                skip.update(range(i+1, i+L))
    tense = "present"
    for t in doc:
        nxt = doc[t.i+1] if t.i+1 < n else None
        if t.lemma_ == "have" and nxt is not None and nxt.lower_ == "to" and nxt.pos_ == "PART":
            forced[t.i] = "MUST"; skip.add(nxt.i)
        if t.lower_ == "going" and nxt is not None and nxt.lower_ == "to" and nxt.pos_ == "PART":
            tense = "future"; skip.update({t.i, nxt.i})

    # (2) النفي
    neg_verbs, neg_modals = set(), set()
    for t in doc:
        if t.dep_ == "neg":
            target = t.head
            modals = [c for c in target.children
                      if c.tag_ == "MD" and c.lemma_.lower() in MODAL_NEG]
            if modals:
                neg_modals.add(modals[0].i)
                continue
            if target.pos_ == "AUX" and target.tag_ != "MD":
                cands = [c for c in target.children
                         if c.pos_ in {"ADJ", "NOUN", "VERB", "PROPN"}]
                if cands:
                    target = cands[0]
            neg_verbs.add(target.i)

    # (3) الزمن
    for t in doc:
        if t.tag_ == "MD" and t.lemma_.lower() in FUTURE_MODALS:
            tense = "future"
        elif tense == "present":
            if t.tag_ == "VBD":
                tense = "past"
            elif t.tag_ == "VBN" and any(c.lemma_ == "have" for c in t.children):
                tense = "past"

    # (4) Gloss لكل كلمة
    glosses = {}
    for t in doc:
        if t.i in skip:
            continue
        if t.i in forced:
            glosses[t.i] = [forced[t.i]]; continue
        if t.dep_ in {"neg", "expl"}:
            continue
        lem = t.lemma_.lower()
        if t.tag_ == "MD":
            if lem in FUTURE_MODALS:
                continue
            g = MODAL_NEG.get(lem) if t.i in neg_modals else MODAL_MAP.get(lem)
            if g:
                glosses[t.i] = [g]
            continue
        g = base_gloss(t)
        if g is None:
            continue
        out = [g]
        if t.i in neg_verbs:
            fused = NEG_FUSION.get(lem)
            out = [fused] if fused else [g, "NOT"]
        glosses[t.i] = out

    order = [t.i for t in doc if t.i in glosses]

    # (5) المفعول قبل الفعل (SOV)
    for v in doc:
        if v.pos_ == "VERB" and v.i in glosses:
            for d in v.children:
                if d.dep_ in {"dobj", "obj"} and d.i > v.i and d.pos_ != "PRON":
                    sub = [x.i for x in d.subtree if x.i in order]
                    if not sub:
                        continue
                    for x in sub:
                        order.remove(x)
                    pos = order.index(v.i)
                    order[pos:pos] = sub

    # (6) الزمن في البداية
    time_set = set()
    for t in doc:
        if t.i not in glosses:
            continue
        lem = t.lemma_.upper()
        if lem in TIME_WORDS or (lem in TIME_UNITS and t.dep_ in {"npadvmod", "tmod"}):
            time_set.add(t.i)
            for c in t.children:
                if c.lemma_.lower() in TIME_MODIFIERS and c.i in glosses:
                    time_set.add(c.i)
    order = [i for i in order if i in time_set] + [i for i in order if i not in time_set]

    # (7) كلمة الاستفهام في النهاية
    wh_set = {t.i for t in doc if t.lower_ in WH_WORDS
              and t.tag_ in WH_TAGS and t.i in glosses and t.i not in forced}
    order = [i for i in order if i not in wh_set] + [i for i in order if i in wh_set]

    # (8) علامة الزمن
    final = [g for i in order for g in glosses[i]]
    if not time_set:
        if tense == "past":
            final.insert(0, PAST_MARKER)
        elif tense == "future":
            final.insert(0, FUTURE_MARKER)

    # (9) نوع الجملة
    is_q = sentence.strip().endswith("?")
    has_wh = bool(wh_set) or any(g == "HOW-ARE-YOU" for g in final)
    stype = ("wh-question" if is_q and has_wh else
             "yes-no-question" if is_q else "statement")

    return {"gloss": " ".join(final), "type": stype,
            "tense": tense, "negation": bool(neg_verbs or neg_modals)}


def split_clauses(sentence):
    """يرجع قائمة (id, connector, position, text). الأولى هي الجملة الرئيسية."""
    doc = nlp(sentence)
    owner = [0] * len(doc)
    info = {0: (None, "main")}
    nxt = 1

    for t in doc:   # الجمل الفرعية
        if (t.lower_ in CLAUSE_CONFIG and t.head.dep_ == "advcl"
                and t.dep_ in {"mark", "advmod"}):
            for x in t.head.subtree:
                owner[x.i] = nxt
            owner[t.i] = -1
            info[nxt] = CLAUSE_CONFIG[t.lower_]
            nxt += 1

    for t in doc:   # العطف بين جملتين مستقلتين
        if t.dep_ == "cc" and t.lower_ in COORD:
            for c in t.head.children:
                has_subj = any(x.dep_ in {"nsubj", "nsubjpass"} for x in c.children)
                if c.dep_ == "conj" and c.pos_ in {"VERB", "AUX"} and has_subj:
                    for x in c.subtree:
                        owner[x.i] = nxt
                    owner[t.i] = -1
                    info[nxt] = (COORD[t.lower_], "after")
                    nxt += 1
                    break

    parts = []
    for cid, (conn, pos) in info.items():
        toks = [t for t in doc if owner[t.i] == cid]
        text = "".join(t.text_with_ws for t in toks).strip(" ,;")
        if text:
            parts.append((cid, conn, pos, text))
    return parts


def sentence_to_gloss(sentence):
    """الدالة الرئيسية: تقسم الجمل الفرعية ثم تحلل كل جزء وتجمعها."""
    sentence = normalize_text(sentence)
    parts = split_clauses(sentence)
    if len(parts) <= 1:
        return analyze(sentence)

    results = {cid: analyze(text) for cid, _, _, text in parts}
    main_tokens = results[0]["gloss"].split() if 0 in results else []
    main_marker = main_tokens[0] if main_tokens and main_tokens[0] in TENSE_MARKERS else None

    before, after = [], []
    for cid, conn, pos, _ in parts:
        if cid == 0:
            continue
        g = results[cid]["gloss"].split()
        if g and g[0] == main_marker:      # لا نكرر علامة الزمن
            g = g[1:]
        block = ([conn] if conn else []) + g
        (before if pos == "before" else after).append(block)

    final = [t for b in before for t in b] + main_tokens + [t for b in after for t in b]
    out = dict(results.get(0, results[parts[0][0]]))
    out["gloss"] = " ".join(final)
    return out


def text_to_gloss(sentence):
    return sentence_to_gloss(sentence)["gloss"]

print("تم تعريف المحوّل ✅")

In [ ]:
# ---- بدائل لما ليس في الـ 2,731 (اقتراحات لغوية، تُراجع بالعين لاحقاً) ----
# التكييف:
SIGN_MAP = {
    "SHE": ["HE"], "ITS": ["HIS"], "THEIR": ["THEY"], "IT": [],      # IT تُحذف
    "STORE": ["SHOP"], "CAN'T": ["CANNOT"],
    "GOOD-MORNING": ["GOOD", "MORNING"], "HOW-ARE-YOU": ["HOW", "YOU"],
    "TONIGHT": ["NIGHT"], "EVENING": ["NIGHT"],
    "THIS": ["THAT"], "THOSE": ["THAT"], "EVERY": ["EACH"],
    "UNLESS": ["IF", "NOT"], "ALTHOUGH": ["BUT"],
    "GOODBYE": ["BYE"], "EXCUSE-ME": ["EXCUSE"],
    "DON'T-UNDERSTAND": ["NOTUNDERSTAND"],
    "MUST-NOT": ["MUST", "NOT"],
    "SHOULD-NOT": ["SHOULD", "NOT"],
}

# ---- الأرقام: normalize_text يحوّل three إلى 3، لكن الداتا فيها THREE ----
DIGIT_WORD = {"0": "ZERO", "1": "ONE", "2": "TWO", "3": "THREE", "4": "FOUR",
              "5": "FIVE", "6": "SIX", "7": "SEVEN", "8": "EIGHT", "9": "NINE"}
NOT_IN_DATASET = {"FIVE", "TEN", "TWELVE", "TWENTY", "THOUSAND", "MILLION"}
SPECIAL_NUM = {"11": ["ELEVEN"], "100": ["ONE", "HUNDRED"]}

def number_to_glosses(tok):
    if tok in SPECIAL_NUM:                                   # 11 و 100
        return SPECIAL_NUM[tok]
    if len(tok) == 1 and DIGIT_WORD[tok] not in NOT_IN_DATASET:
        return [DIGIT_WORD[tok]]                             # 0-9 ما عدا 5
    return ["fs-" + tok]                                     # 25 → بطاقة نصية "25"

def adapt_gloss(gloss_str):
    out = []
    for t in gloss_str.split():
        if t.startswith("fs-"):
            out.append(t)
        elif t.isdigit():
            out += number_to_glosses(t)
        else:
            out += SIGN_MAP.get(t, [t])
    return " ".join(out)

# الدالة الرئيسية الآن تشمل التكييف؛ والنسخة الخام تبقى للتقارير
def text_to_gloss_raw(sentence):
    return sentence_to_gloss(sentence)["gloss"]

def text_to_gloss(sentence):
    return adapt_gloss(text_to_gloss_raw(sentence))

for s in ["She goes to the store tonight.", "I'm gonna eat three apples tomorrow.",
          "Good morning.", "How are you?", "I have twenty five dollars."]:
    print(f"{s:40} → {text_to_gloss(s)}")

In [ ]:
# ---- بدائل الكلمات الناقصة + العبارات المركبة ----
SIGN_MAP.update({
    "AFRAID": ["SCARED"], "SPEAK": ["TALK"], "JOB": ["WORK"], "HAND": ["HANDS"],
    "O'CLOCK": ["CLOCK"], "CROWDED": ["FULL"],
    "HOW-MUCH": ["HOW", "MUCH"], "HOW-MANY": ["HOW", "MANY"], "WHAT-TIME": ["WHAT", "TIME"],
    # "FORGET": ["FORGETFUL"],     # أزل التعليق بعد أن تشاهد مقطع FORGETFUL وتجده مناسباً
})
PHRASES.update({("how", "much"): "HOW-MUCH", ("how", "many"): "HOW-MANY",
                ("what", "time"): "WHAT-TIME"})

# ---- كلمات يعرفها البنك بالاسم: الأشهر واللغات تُعرض إشارة لا تهجئة ----
SIGN_VOCAB = set(LOOKUP) if "LOOKUP" in globals() else set()

def base_gloss(t):
    w, lem = t.lower_, t.lemma_.lower()
    if w == "her" and t.tag_ == "PRP$":                     # ملكية: her family
        return "HIS"
    if w in PRONOUNS:
        return PRONOUNS[w]
    if w in WH_WORDS and t.tag_ in WH_TAGS:
        return w.upper()
    if w == "left" and t.i > 0 and t.doc[t.i - 1].lower_ in {
            "turn", "turned", "turning", "go", "went", "going", "the", "to", "on", "your", "my", "at"}:
        return "LEFT"                                       # تصحيح اشتقاق spaCy
    if lem.upper() in TIME_WORDS:
        return lem.upper()
    if (t.pos_ == "PROPN" and t.ent_type_ in {"DATE", "LANGUAGE", "NORP"}
            and re.sub(r"[^A-Z0-9]", "", t.text.upper()) in SIGN_VOCAB):
        return t.text.upper()                               # JUNE, ENGLISH ...
    if t.pos_ == "PROPN" or t.ent_type_ in NAME_ENTS:
        return "fs-" + t.text.upper()
    if t.pos_ == "DET" and w in KEEP_DET:
        return w.upper()
    if t.pos_ in DROP_POS or w in DROP_WORDS:
        return None
    if t.like_num:
        return t.text.upper()
    return LEXICON.get(w) or LEXICON.get(lem) or lem.upper()

# ---- الدالة الرئيسية: كلمات الاستفهام المركبة تذهب إلى آخر الجملة ----
MOVE_END = {"HOW-MUCH", "HOW-MANY", "WHAT-TIME"}
def text_to_gloss(sentence):
    toks = text_to_gloss_raw(sentence).split()
    moved = [t for t in toks if t in MOVE_END]
    toks = [t for t in toks if t not in MOVE_END] + moved
    return adapt_gloss(" ".join(toks))

for s in ["She is cooking dinner for her family.", "How much does this cost?",
          "What time does the bus leave?", "Turn left at the next street.",
          "My friend is getting married in June.", "She wants to learn English.",
          "I am afraid of dogs.", "Please speak slowly."]:
    print(f"{s:42} → {text_to_gloss(s)}")

In [ ]:
SIGN_VOCAB = set(LOOKUP)

In [ ]:
for s in ["What time is it?", "I don't understand.", "Which book do you want?"]:
    print(f"{s:28} → {text_to_gloss(s)}")

In [ ]:
TESTS = [
    ("I am going to the store tomorrow.", "TOMORROW ME GO STORE"),
    ("My brother lives in London.",       "MY BROTHER LIVE fs-LONDON"),
    ("I don't like apples.",              "ME APPLE DON'T-LIKE"),
    ("What is your name?",                "YOUR NAME WHAT"),
    ("Where do you live?",                "YOU LIVE WHERE"),
    ("I am not happy.",                   "ME HAPPY NOT"),
    ("I will go home.",                   "FUTURE ME GO HOME"),
    ("I can't swim.",                     "ME CAN'T SWIM"),
    ("I have to go.",                     "ME MUST GO"),
    ("I want to eat pizza.",              "ME WANT PIZZA EAT"),
    ("Did you eat breakfast yesterday?",  "YESTERDAY YOU BREAKFAST EAT"),
    ("I will visit my mom next week.",    "NEXT WEEK ME MY MOTHER VISIT"),
    ("Thank you.",                        "THANK-YOU"),
]

ok = 0
for s, exp in TESTS:
    r = sentence_to_gloss(s)
    good = r["gloss"] == exp
    ok += good
    print(("✅" if good else "❌"), s, "→", r["gloss"], f"[{r['type']}]")
    if not good:
        print("     المتوقع:", exp)
print(f"\nالنجاح: {ok}/{len(TESTS)}")

print("\n--- تجارب بلا نتيجة متوقعة ---")
for s in ["I'm gonna eat three apples tomorrow.",
          "Um, I I wanna eat pizza.",
          "If it rains, I will stay home.",
          "I stayed home because it rained.",
          "I like pizza but he likes pasta.",
          "Going to the store tomorrow."]:
    print(f"{s:40} → {text_to_gloss(s)}")

In [ ]:
import pandas as pd
from google.colab import files

SENTENCES = [
    "I like apples.", "I don't like pizza.", "What is your name?",
    "Where do you live?", "I am going to the store tomorrow.",
    "My brother lives in London.", "I will visit my mother next week.",
    "Did you eat breakfast yesterday?", "I want to eat pizza.",
    "I am not happy.", "I can't swim.", "I have to go home.",
    "Thank you.", "Good morning.", "My friend works at Google.",
    "Do you understand?", "She is a teacher.", "We need water.",
    "Please help me.", "How are you?",
]

rows = []
for s in SENTENCES:
    r = sentence_to_gloss(s)
    rows.append({"sentence": s, "gloss": r["gloss"], "type": r["type"]})
    print(f"{s:40} → {r['gloss']}   [{r['type']}]")

pd.DataFrame(rows).to_csv("sentences_glosses.csv", index=False)
files.download("sentences_glosses.csv")

In [ ]:
#فك ملف بنك الاشارات
from google.colab import drive
drive.mount("/content/drive")

ZIP = "/content/drive/MyDrive/sign_bank.zip"      # عدّل المسار إن وضعته في مجلد
!rm -rf /content/sign_bank && mkdir -p /content/sign_bank
!unzip -q "$ZIP" -d /content/sign_bank
!ls /content/sign_bank
!ls /content/sign_bank/clips | wc -l

In [ ]:
import os, re, subprocess, pandas as pd
from PIL import Image, ImageDraw, ImageFont

BANK = "/content/sign_bank"
norm = lambda g: re.sub(r"[^A-Z0-9]", "", str(g).upper())
base = lambda g: re.sub(r"\d+$", "", norm(g))

lk = pd.read_csv(f"{BANK}/lookup.csv")
LOOKUP = {k: f"{BANK}/clips/{f}" for k, f in zip(lk["key"], lk["file"])}
print("مفاتيح:", len(LOOKUP), "| WHAT →", os.path.basename(LOOKUP["WHAT"]))   # المتوقع WHAT1.mp4
CHOICE = {"CAN": "CAN1", "HOW": "HOW1", "NIGHT": "NIGHT1", "THEY": "THEY1", "WHAT": "WHAT1" , "EAT":"EAT1" ,"PIZZA":"PIZZA1" ,"WANT":"WANT1" ,"LIVE":"LIVE1"}
for b, g in CHOICE.items():
    LOOKUP[b] = f"{BANK}/clips/{norm(g)}.mp4"

def find_clip(tok):
    if tok.startswith("fs-"):
        return None                                  # الأسماء: بطاقة نصية
    return LOOKUP.get(norm(tok)) or LOOKUP.get(base(tok))

FONT = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
def make_card(text, secs=1.0):
    os.makedirs(f"{WORK}/cards", exist_ok=True)
    out = f"{WORK}/cards/{norm(text) or 'card'}.mp4"
    if os.path.exists(out):
        return out
    img = Image.new("RGB", (480, 360), (20, 20, 20))
    d = ImageDraw.Draw(img)
    size = 72
    while True:
        try:
            f = ImageFont.truetype(FONT, size)
        except Exception:
            f = ImageFont.load_default(); break
        if d.textlength(text, font=f) <= 440 or size <= 20:
            break
        size -= 4
    b = d.textbbox((0, 0), text, font=f)
    d.text(((480 - (b[2] - b[0])) / 2 - b[0], (360 - (b[3] - b[1])) / 2 - b[1]),
           text, font=f, fill=(255, 255, 255))
    png = out.replace(".mp4", ".png")
    img.save(png)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", png,
                    "-t", str(secs), "-r", "25", "-vf", "scale=480:360",
                    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "26", out], check=True)
    return out

def gloss_to_clips(gloss):
    clips, missing = [], []
    for tok in gloss.split():
        p = find_clip(tok)
        if p is None:
            missing.append(tok)
            p = make_card(tok.replace("fs-", ""))
        clips.append(p)
    return clips, missing

def merge_clips(clips, out):
    lst = out + ".txt"
    with open(lst, "w") as f:
        for c in clips:
            f.write(f"file '{c}'\n")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
                    "-i", lst, "-c:v", "libx264", "-crf", "26", "-preset", "veryfast",
                    "-pix_fmt", "yuv420p", "-an", out], check=True)
    return out

def sentences_to_video(sentences, out=f"{WORK}/result.mp4"):
    clips, missing = [], []
    for s in sentences:
        g = text_to_gloss(s)
        print(f"{s}  →  {g}")
        c, m = gloss_to_clips(g)
        clips += c; missing += m
    assert clips, "لا توجد مقاطع"
    merge_clips(clips, out)
    print("بطاقات نصية بدل إشارة:", missing or "لا شيء")
    return out

def speech_to_sign(audio_path, out=f"{WORK}/result.mp4"):
    text = clean_text(transcribe_audio(audio_path))
    print("النص:", text)
    return sentences_to_video(split_sentences(text), out)

In [ ]:
SUFFIXES = ["LY", "Y", "ED", "ING", "ER", "EST", "ES", "S"]
FALLBACK_LOG = {}

def candidates(tok):
    t = norm(tok)
    out = [t]
    for suf in SUFFIXES:
        if t.endswith(suf) and len(t) - len(suf) >= 3:
            stem = t[:-len(suf)]
            out += [stem, stem + "E"]
    return out

def find_clip(tok):
    if tok.startswith("fs-"):
        return None
    for i, c in enumerate(candidates(tok)):
        p = LOOKUP.get(c) or LOOKUP.get(base(c))
        if p:
            if i > 0:
                FALLBACK_LOG[tok] = c           # استُخدم الاحتياط: راجعه
            return p
    return None

print("find_clip الجديدة جاهزة")

In [ ]:
import hashlib

def caption_clip(clip, label):
    """يكتب اسم الإشارة أسفل المقطع."""
    os.makedirs(f"{WORK}/cap", exist_ok=True)
    out = f"{WORK}/cap/{hashlib.md5(f'{clip}|{label}'.encode()).hexdigest()[:10]}.mp4"
    if os.path.exists(out):
        return out
    txt = out.replace(".mp4", ".txt")
    open(txt, "w").write(label)
    vf = (f"drawtext=fontfile={FONT}:textfile={txt}:fontsize=26:fontcolor=white:"
          "box=1:boxcolor=black@0.6:boxborderw=8:x=(w-text_w)/2:y=h-text_h-16")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", clip, "-vf", vf, "-an",
                    "-c:v", "libx264", "-crf", "26", "-preset", "veryfast",
                    "-pix_fmt", "yuv420p", out], check=True)
    return out

def pause_clip(clip, secs=0.4):
    """وقفة قصيرة من أول لقطة في المقطع (وضعية الراحة)."""
    os.makedirs(f"{WORK}/cards", exist_ok=True)
    out = f"{WORK}/cards/pause_{secs}_{os.path.basename(clip)}"
    if os.path.exists(out):
        return out
    png = out.replace(".mp4", ".png")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", clip, "-frames:v", "1", png], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-loop", "1", "-i", png, "-t", str(secs),
                    "-r", "25", "-vf", "scale=480:360", "-c:v", "libx264", "-pix_fmt", "yuv420p",
                    "-crf", "26", out], check=True)
    return out

def sentences_to_video(sentences, out=f"{WORK}/result.mp4", pause=0.9, captions=True):
    clips, missing = [], []
    for n, s in enumerate(sentences):
        g = text_to_gloss(s)
        print(f"{s}  →  {g}")
        seg, first = [], None
        for tok in g.split():
            p = find_clip(tok)
            if p is None:
                missing.append(tok)
                p = make_card(tok.replace("fs-", ""))      # البطاقة تحوي النص أصلاً
                seg.append(p)
            else:
                seg.append(caption_clip(p, tok) if captions else p)
            first = first or p
        if not seg:
            continue
        if n > 0 and pause:
            clips.append(pause_clip(first, pause))
        clips += seg
    assert clips, "لا توجد مقاطع"
    merge_clips(clips, out)
    print("بطاقات نصية بدل إشارة:", missing or "لا شيء")
    return out

v = sentences_to_video(["I like apples.", "What time is it?", "I don't understand."])
Video(v, embed=True, width=480)

In [ ]:
def variants_of(tok):
    b = base(tok)
    return sorted(f[:-4] for f in os.listdir(f"{BANK}/clips") if re.fullmatch(rf"{b}\d+\.mp4", f))

demo = ["I like apples.", "I want to eat pizza.", "Where do you live?",
        "My brother works at school.", "Can you help me?", "What is your name?"]
need_check = {}
for s in demo:
    for tok in text_to_gloss(s).split():
        v = variants_of(tok) if not tok.startswith("fs-") else []
        if len(v) > 1:
            need_check[tok] = v
print(need_check)

In [ ]:
from IPython.display import Video, display
for k in ["WANT1", "WANT2", "PIZZA1", "PIZZA2", "EAT1", "EAT2", "LIVE1", "LIVE2", "CAN1", "CAN2" , "WHAT1" ,"WHAT2"]:
    print(k)
    display(Video(f"{BANK}/clips/{k}.mp4", embed=True, width=240))

In [ ]:
# من النص لفيديو
from IPython.display import Video
v = sentences_to_video(["I like apples.", "My brother lives in London.", "What is your name?"])
Video(v, embed=True, width=480)

In [ ]:
# من الصوت للفيديو مباشرة
v = speech_to_sign(audio_file)
Video(v, embed=True, width=480)

In [ ]:
NEW = [
    "I am hungry.", "Where is the bathroom?", "My sister is sick.",
    "I need help.", "We are going to school.", "Do you like coffee?",
    "He doesn't know my name.", "She will call you tomorrow.",
    "I am learning sign language.", "The doctor is here.",
    "Can you help me?", "I don't understand.",
    "My father works in a hospital.", "What time is it?", "I love my family.",
]

tot, missing_all = 0, []
for s in NEW:
    g = text_to_gloss(s)
    toks = g.split()
    m = [t for t in toks if find_clip(t) is None]      # تشمل الأسماء fs- لأنها بطاقات نصية
    tot += len(toks); missing_all += m
    print("✅" if not m else "❌", s, "→", g, "| بطاقة نصية:", m or "-")

hit = tot - len(missing_all)
print(f"\nإشارات حقيقية: {hit}/{tot} = {hit/tot:.1%}")
print("الكلمات التي صارت بطاقات:", sorted(set(missing_all)))

In [ ]:
import cv2, math, numpy as np, pandas as pd, os
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video

man  = pd.read_csv(f"{BANK}/bank_manifest.csv")
keys = man["key"].tolist()

def probe(key):
    cap = cv2.VideoCapture(f"{BANK}/clips/{key}.mp4")
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames, diffs, prev = [], [], None
    for i in np.linspace(0, max(n - 1, 0), 8).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if not ok:
            continue
        frames.append(fr)
        g = cv2.cvtColor(cv2.resize(fr, (80, 60)), cv2.COLOR_BGR2GRAY).astype(np.float32)
        if prev is not None:
            diffs.append(float(np.abs(g - prev).mean()))
        prev = g
    cap.release()
    if not frames:
        return key, None, dict(key=key, dur=0, motion=0, bright=0)
    mid = frames[len(frames) // 2]
    thumb = cv2.cvtColor(cv2.resize(mid, (160, 120)), cv2.COLOR_BGR2RGB)
    return key, thumb, dict(key=key, dur=n / fps, motion=float(np.mean(diffs)) if diffs else 0,
                            bright=float(mid.mean()))

with ThreadPoolExecutor(4) as ex:
    out = list(ex.map(probe, keys))

THUMBS = {k: t for k, t, _ in out}
M = pd.DataFrame([m for _, _, m in out]).set_index("key")
print(M.describe().round(2))

In [ ]:
M["flag"] = ""
M.loc[M.dur < 1.0, "flag"] += "قصير "
M.loc[M.dur > 4.0, "flag"] += "طويل "
M.loc[M.motion <= M["motion"].quantile(0.03), "flag"] += "ساكن "
M.loc[(M.bright < 50) | (M.bright > 210), "flag"] += "إضاءة "
M.loc[[k for k, t in THUMBS.items() if t is None], "flag"] += "تالف "

flagged = M[M.flag != ""].index.tolist()
flagged_set = set(flagged)
print("مشبوه:", len(flagged), "من", len(M))
print(M[M.flag != ""].flag.value_counts().head(8))

In [ ]:
FONT_S = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
PER = 60

def sheet(ks, cols=10, tw=160, th=120):
    rows = math.ceil(len(ks) / cols)
    f = ImageFont.truetype(FONT_S, 12)
    img = Image.new("RGB", (cols * tw, rows * th), (30, 30, 30))
    d = ImageDraw.Draw(img)
    for i, k in enumerate(ks):
        x, y = (i % cols) * tw, (i // cols) * th
        if THUMBS.get(k) is not None:
            img.paste(Image.fromarray(THUMBS[k]), (x, y))
        d.rectangle([x, y, x + tw, y + 15], fill=(0, 0, 0))
        d.text((x + 3, y + 1), k[:22], font=f, fill=(255, 255, 0) if k in flagged_set else (255, 255, 255))
    return img

def show_page(n, only_flagged=False):
    ks = flagged if only_flagged else keys
    print(f"صفحة {n + 1} من {math.ceil(len(ks) / PER)}")
    display(sheet(ks[n * PER:(n + 1) * PER]))

def show_clip(k):
    print(k, M.loc[k, "flag"] or "-")
    display(Video(f"{BANK}/clips/{k}.mp4", embed=True, width=240))

In [ ]:
def get_font(size=12):
    for p in [FONT,
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"]:
        try:
            return ImageFont.truetype(p, size)
        except OSError:
            pass
    return ImageFont.load_default()

def sheet(ks, cols=10, tw=160, th=120):
    rows = math.ceil(len(ks) / cols)
    f = get_font(12)
    img = Image.new("RGB", (cols * tw, rows * th), (30, 30, 30))
    d = ImageDraw.Draw(img)

    for i, k in enumerate(ks):
        x, y = (i % cols) * tw, (i // cols) * th
        if THUMBS.get(k) is not None:
            img.paste(Image.fromarray(THUMBS[k]), (x, y))
        d.rectangle([x, y, x + tw, y + 15], fill=(0, 0, 0))
        d.text((x + 3, y + 1), k[:22], font=f,
               fill=(255, 255, 0) if k in flagged_set else (255, 255, 255))
    return img

show_page(0, only_flagged=True)

In [ ]:
show_page(1, only_flagged=True)

In [ ]:
light = M[M.flag.str.contains("إضاءة")].index.tolist()
print(light, M.loc[light, ["dur", "bright"]].round(1).to_dict("index"))
display(sheet(light, cols=3))

In [ ]:
low = M.sort_values("motion").head(12)
print(low[["dur", "motion"]].round(2))
display(sheet(low.index.tolist(), cols=6))
for k in low.index[:3]:
    show_clip(k)

In [ ]:
show_page(0, only_flagged=True)
show_page(1, only_flagged=True)

In [ ]:
import random
for n in random.sample(range(math.ceil(len(keys) / PER)), 10):
    show_page(n)

In [ ]:
TEST40 = [
    "Good morning, how are you today?", "I want to buy some bread.",
    "The weather is cold and rainy.", "My teacher gave me homework.",
    "Please open the window.", "Where is the nearest hospital?",
    "I lost my phone yesterday.", "She is cooking dinner for her family.",
    "We will travel to Paris next month.", "Do you have any brothers or sisters?",
    "I don't like cold water.", "He cannot come to the meeting.",
    "The children are playing in the garden.", "I need to see a doctor.",
    "How much does this cost?", "What time does the bus leave?",
    "My grandmother lives in a small village.", "I am afraid of dogs.",
    "Can I borrow your pen?", "They study computer science at university.",
    "It is my birthday today.", "I forgot my keys at home.",
    "Please speak slowly.", "I am very tired and want to sleep.",
    "The movie starts at eight o'clock.", "He drinks coffee every morning.",
    "I have two cats and one dog.", "Where did you buy that shirt?",
    "My friend is getting married in June.", "I am sorry, I am late.",
    "We need to clean the kitchen.", "She wants to learn English.",
    "The train was very crowded.", "Turn left at the next street.",
    "I would like some tea, please.", "Are you angry with me?",
    "Tomorrow I will visit my uncle.", "He lost his job last year.",
    "Wash your hands before eating.", "I think this is a good idea.",
]

rows = []
for s in TEST40:
    g = text_to_gloss(s)
    toks = g.split()
    names = [t for t in toks if t.startswith("fs-")]
    miss  = [t for t in toks if not t.startswith("fs-") and find_clip(t) is None]
    rows.append(dict(sentence=s, gloss=g, n=len(toks) - len(names), miss=miss, names=names))
R = pd.DataFrame(rows)

tot, nm = R.n.sum(), R.miss.map(len).sum()
print(f"إشارات حقيقية: {tot - nm}/{tot} = {(tot - nm) / tot:.1%}")
print(f"جمل مغطاة كاملاً: {(R.miss.map(len) == 0).sum()}/{len(R)}")
print("\nالناقص (كلمات تحولت بطاقات):")
for w, c in pd.Series([w for m in R.miss for w in m]).value_counts().items():
    print(f"  {w:14} × {c}")
print("\nجمل فيها نقص:")
for _, r in R[R.miss.map(len) > 0].iterrows():
    print(f"  {r.sentence}\n     → {r.gloss}\n     ناقص: {r.miss}")

In [ ]:
man = pd.read_csv(f"{BANK}/bank_manifest.csv")
G = man["Gloss"].tolist()
probe = {"AFRAID": ["SCARE", "FEAR", "AFRAID", "FRIGHT"], "FORGET": ["FORGET", "FORGOT"],
         "SPEAK": ["TALK", "SPEAK", "SAY", "SPEECH"], "O'CLOCK": ["CLOCK", "OCLOCK"],
         "CROWDED": ["CROWD", "FULL", "MANY"], "JOB": ["JOB", "WORK"],
         "HAND": ["HAND"], "GARDEN": ["GARDEN", "YARD", "FLOWER", "PLANT"]}
for w, cands in probe.items():
    hits = [g for g in G if any(norm(g).startswith(c) for c in cands)]
    print(f"{w:9} →", hits[:12])

In [ ]:
for _, r in R[R.miss.map(len) == 0].iterrows():
    print(f"{r.sentence:46} → {r.gloss}")

In [ ]:
for w in ["JUNE", "ENGLISH", "LEFT", "RIGHT", "SLOW", "RAIN", "FORGETFUL", "TALK",
          "SCARED", "WORK", "HANDS", "CLOCK", "FULL", "MUCH", "MANY", "HIS"]:
    print(f"{w:10}", w in LOOKUP or base(w) in LOOKUP)

In [ ]:
import cv2, numpy as np, subprocess, os, time
from concurrent.futures import ThreadPoolExecutor

TRIM_DIR = f"{BANK}/clips_trim"
os.makedirs(TRIM_DIR, exist_ok=True)

def motion_profile(path):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    prev, d = None, []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        g = cv2.GaussianBlur(cv2.cvtColor(cv2.resize(fr, (160, 120)), cv2.COLOR_BGR2GRAY),
                             (5, 5), 0).astype(np.float32)
        d.append(0.0 if prev is None else float(np.abs(g - prev).mean()))
        prev = g
    cap.release()
    return np.array(d), fps

def trim_range(path, rel=0.30, pad=3, max_cut=0.40, min_len=0.8):
    d, fps = motion_profile(path); n = len(d)
    if n < 10:
        return 0.0, None
    s = np.convolve(d, np.ones(3) / 3, mode="same")
    thr = max(0.5, rel * np.percentile(s, 95))
    idx = np.where(s > thr)[0]
    if len(idx) == 0:
        return 0.0, None
    a = min(max(0, idx[0] - pad), int(n * max_cut))
    b = max(min(n - 1, idx[-1] + pad), int(n * (1 - max_cut)))
    if (b - a + 1) / fps < min_len:
        return 0.0, None
    return a / fps, (b + 1) / fps

def trim_one(key, **kw):
    src, dst = f"{BANK}/clips/{key}.mp4", f"{TRIM_DIR}/{key}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 2000:
        return key, True
    a, b = trim_range(src, **kw)
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ss", f"{a:.3f}"]
    if b is not None:
        cmd += ["-to", f"{b:.3f}"]
    cmd += ["-an", "-c:v", "libx264", "-crf", "26", "-preset", "veryfast", "-pix_fmt", "yuv420p", dst]
    r = subprocess.run(cmd, capture_output=True)
    return key, (r.returncode == 0 and os.path.exists(dst) and os.path.getsize(dst) > 2000)

# تجربة على 12 مقطعاً: الأصلي ← بعد القص (ثانية)
import random
random.seed(1)
sample = random.sample(keys, 12)
for k in sample:
    d, fps = motion_profile(f"{BANK}/clips/{k}.mp4")
    a, b = trim_range(f"{BANK}/clips/{k}.mp4")
    print(f"{k:14} {len(d)/fps:.1f}s → {((b or len(d)/fps) - a):.1f}s   (قص {a:.2f}s من البداية)")

In [ ]:
from IPython.display import Video, display
for k in sample[:3]:
    trim_one(k)
    print(k, "← الأصلي"); display(Video(f"{BANK}/clips/{k}.mp4", embed=True, width=240))
    print(k, "← بعد القص"); display(Video(f"{TRIM_DIR}/{k}.mp4", embed=True, width=240))

In [ ]:
import matplotlib.pyplot as plt

def dur(p):
    c = cv2.VideoCapture(p); n = c.get(cv2.CAP_PROP_FRAME_COUNT); f = c.get(cv2.CAP_PROP_FPS) or 25
    c.release(); return n / f

for k in sample[:6]:
    trim_one(k)
    print(f"{k:14} أصلي {dur(f'{BANK}/clips/{k}.mp4'):.2f}s → بعد القص {dur(f'{TRIM_DIR}/{k}.mp4'):.2f}s")

fig, axs = plt.subplots(1, 3, figsize=(15, 3))
for ax, k in zip(axs, sample[:3]):
    d, fps = motion_profile(f"{BANK}/clips/{k}.mp4")
    s = np.convolve(d, np.ones(3) / 3, mode="same")
    thr = max(0.5, 0.20 * np.percentile(s, 95))
    a, b = trim_range(f"{BANK}/clips/{k}.mp4")
    t = np.arange(len(d)) / fps
    ax.plot(t, s); ax.axhline(thr, color="r", ls="--", label="threshold")
    ax.axvline(a, color="g"); ax.axvline(b or t[-1], color="g")
    ax.set_title(k); ax.set_xlabel("sec"); ax.legend()
plt.show()

In [ ]:
def trim_to(key, outdir, rel, pad):
    os.makedirs(outdir, exist_ok=True)
    src, dst = f"{BANK}/clips/{key}.mp4", f"{outdir}/{key}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 2000:
        return dst
    a, b = trim_range(src, rel=rel, pad=pad)
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ss", f"{a:.3f}"]
    if b is not None:
        cmd += ["-to", f"{b:.3f}"]
    cmd += ["-an", "-c:v", "libx264", "-crf", "26", "-preset", "veryfast", "-pix_fmt", "yuv420p", dst]
    subprocess.run(cmd, check=True)
    return dst

def video_with(name, rel, pad, sentences):
    global LOOKUP
    saved = dict(LOOKUP)
    orig = {k: p.replace("/clips_trim/", "/clips/") for k, p in saved.items()}   # المقاطع الأصلية
    LOOKUP = dict(orig)
    needed = set()
    for s in sentences:
        for tok in text_to_gloss(s).split():
            p = find_clip(tok)
            if p:
                needed.add(p)
    mapping = {}
    if rel is not None:
        mapping = {p: trim_to(os.path.basename(p)[:-4], f"{BANK}/clips_{name}", rel, pad) for p in needed}
    LOOKUP = {k: mapping.get(p, p) for k, p in orig.items()}
    try:
        return sentences_to_video(sentences, out=f"{WORK}/cmp_{name}.mp4", captions=False)
    finally:
        LOOKUP = saved

SENT = ["I want to eat pizza.", "Where do you live?"]
for name, rel, pad in [("orig", None, None), ("t20", 0.20, 3), ("t35", 0.35, 1)]:
    v = video_with(name, rel, pad, SENT)
    print(name, "→ المدة:", round(dur(v), 1), "ث")
    display(Video(v, embed=True, width=400))

In [ ]:
import cv2, time, numpy as np, subprocess, os, pandas as pd
from concurrent.futures import ThreadPoolExecutor

def motion_profile(path):
    cap = cv2.VideoCapture(path); fps = cap.get(cv2.CAP_PROP_FPS) or 25
    prev, d = None, []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        g = cv2.GaussianBlur(cv2.cvtColor(cv2.resize(fr, (160, 120)), cv2.COLOR_BGR2GRAY),
                             (5, 5), 0).astype(np.float32)
        d.append(0.0 if prev is None else float(np.abs(g - prev).mean()))
        prev = g
    cap.release()
    return np.array(d), fps

def trim_range(path, rel=0.20, pad=3, max_cut=0.40, min_len=0.8):
    d, fps = motion_profile(path); n = len(d)
    if n < 10:
        return 0.0, None
    s = np.convolve(d, np.ones(3) / 3, mode="same")
    thr = max(0.5, rel * np.percentile(s, 95))
    idx = np.where(s > thr)[0]
    if len(idx) == 0:
        return 0.0, None
    a = min(max(0, idx[0] - pad), int(n * max_cut))
    b = max(min(n - 1, idx[-1] + pad), int(n * (1 - max_cut)))
    if (b - a + 1) / fps < min_len:
        return 0.0, None
    return a / fps, (b + 1) / fps

def trim_to(key, outdir, rel, pad):
    os.makedirs(outdir, exist_ok=True)
    src, dst = f"{BANK}/clips/{key}.mp4", f"{outdir}/{key}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 2000:
        return dst
    a, b = trim_range(src, rel=rel, pad=pad)
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ss", f"{a:.3f}"]
    if b is not None:
        cmd += ["-to", f"{b:.3f}"]
    cmd += ["-an", "-c:v", "libx264", "-crf", "26", "-preset", "veryfast", "-pix_fmt", "yuv420p", dst]
    subprocess.run(cmd, check=True)
    return dst

keys = list(pd.read_csv(f"{BANK}/bank_manifest.csv")["key"])

t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    done = list(ex.map(lambda k: trim_to(k, f"{BANK}/clips_t35", 0.35, 1), keys))
print(f"تم قص {len(done)} مقطعاً في {(time.time()-t0)/60:.1f} دقيقة")

def use_t35():
    n = 0
    for k, p in list(LOOKUP.items()):
        q = f"{BANK}/clips_t35/{os.path.basename(p)}"
        if os.path.exists(q):
            LOOKUP[k] = q; n += 1
    print("مفاتيح تشير إلى المقاطع المقصوصة:", n, "من", len(LOOKUP))

use_t35()

In [ ]:
v = sentences_to_video(["I want to eat pizza.", "Where do you live?", "Thank you."])
Video(v, embed=True, width=480)

In [ ]:
!cd /content/sign_bank && zip -qr /content/drive/MyDrive/sign_bank_t35.zip clips_t35
!ls -l /content/drive/MyDrive/sign_bank_t35.zip